# resnet-stem — worked example 1: Assemble the four-op ResNet stem

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `resnet-stem`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The ResNet stem is a fixed four-op recipe: a 7x7 stride-2 conv (padding 3, no bias), BatchNorm, ReLU, then a 3x3 stride-2 maxpool (padding 1). It lifts 3 input channels to 64 and downsamples spatially by 4x in one block, turning a 224x224 image into a 56x56 feature map.

## Worked solution

We wrap the four ops in an `nn.Sequential` in the canonical order. The conv uses `bias=False` because the following BatchNorm has its own learnable shift, making a conv bias redundant. The conv halves spatial size: `(224 + 2*3 - 7)//2 + 1 = 112`. BatchNorm and ReLU keep the shape. The maxpool halves again: `(112 + 2*1 - 3)//2 + 1 = 56`. We run a `(1, 3, 224, 224)` input and print the output shape `(1, 64, 56, 56)` to confirm the 4x downsample and channel lift.

In [ ]:
import torch.nn as nn


def build_stem():
    return nn.Sequential(
        nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
        nn.BatchNorm2d(64),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
    )


stem = build_stem()
x = t.zeros(1, 3, 224, 224)
stem.eval()
out = stem(x)
print('output shape:', tuple(out.shape))
print('num children:', len(list(stem.children())))